## Задание 1. 
Разработать или взять в готовом виде код трансформера, способного к генерации текста после обучения. Готовый пример трансформера для библиотеки pytorch имеется в файле task7_transformer.py, приложенном к описанию работы. Некоторые другие примеры доступного для использования кода:
Для библиотеки pytorch:
https://github.com/pytorch/examples/tree/main/word_language_model

https://pytorch.org/tutorials/beginner/transformer_tutorial.html (использует устаревший модуль torchtext, может не сработать)

Для библиотеки tensorflow:
https://www.tensorflow.org/text/tutorials/transformer

Воспользуемся ai-forever/rugpt3small_based_on_gpt2 (https://huggingface.co/ai-forever/rugpt3small_based_on_gpt2)



## Задание 2. 
Загрузить пример текста для обучения трансформера или использовать готовый пример. В примере по умолчанию используется набор данных WikiText2; файл с ним может быть изменён, сокращён или заменён при необходимости. Так как обучение на полном тексте WikiText2 может быть весьма долгим (порядка нескольких часов или дней), допускается
урезание используемого файла по усмотрению студента.

В качестве dataset возьмем прозу 19 века (https://huggingface.co/datasets/Imperius/ru-classic)

## Задание 3. 
Провести обучение трансформера на выбранных данных, после чего сгенерировать на нём несколько отрывков текста. Сделать вывод о качестве генерации; оценить влияние на неё параметров трансформера, в том числе количества головок в механизме самовнимания, длины контекста и степени случайности генерации.

In [10]:
import math
import random
import time
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import json
import zstandard
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup

In [2]:
model_id = "ai-forever/rugpt3small_based_on_gpt2"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32
).to("cuda")

# Проверка
inputs = tokenizer("Нейронные сети в физике применяются для", return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
d:\miniconda\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\darkitime\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  551MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

d:\miniconda\envs\ml\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\darkitime\.cache\huggingface\hub\models--ai-forever--rugpt3small_based_on_gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  551MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Нейронные сети в физике применяются для передачи данных, как в компьютерной, так и в промышленной сетях, и в различных отраслях промышленности, в том числе и в телекоммуникационных сетях.  Термины "фазное" и "электронное" относятся к системам, построенным на


In [6]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [7]:
@torch.no_grad()
def evaluate_loss(model, dataloader, device) -> float:
    model.eval()
    total_loss = 0.0
    total_steps = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        total_loss += outputs.loss.item()
        total_steps += 1
    model.train()
    return total_loss / max(total_steps, 1)

In [8]:
@torch.no_grad()
def generate_samples(model, tokenizer, prompts, device, temperature=0.8, top_p=0.9, max_new_tokens=60):
    model.eval()
    results = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_len = inputs["input_ids"].shape[1]
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        gen_only = tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True)
        full_text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        results.append((prompt, gen_only.strip(), full_text))
    model.train()
    return results

In [11]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

model_id = "ai-forever/rugpt3small_based_on_gpt2"
max_length = 256
batch_size = 4
grad_accum_steps = 4
epochs = 2
lr = 5e-5
weight_decay = 0.01

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32).to(device)

# Dataset
archive_path = hf_hub_download(
    repo_id="IlyaGusev/ru_turbo_alpaca",
    filename="ru_turbo_alpaca.jsonl.zst",
    repo_type="dataset",
)

records = []
max_samples = 2000

with open(archive_path, "rb") as f:
    dctx = zstandard.ZstdDecompressor()
    with dctx.stream_reader(f) as reader:
        text_data = reader.read().decode("utf-8")
        for line in text_data.splitlines():
            if not line.strip():
                continue
            records.append(json.loads(line))
            if len(records) >= max_samples:
                break

raw_ds = Dataset.from_list(records)
print("Загружено примеров:", len(raw_ds), "| Колонки:", raw_ds.column_names)

def format_example(ex):
    ctx = f"\nКонтекст: {ex['input']}" if ex.get("input") else ""
    return {"text": f"Задание: {ex['instruction']}{ctx}\nОтвет: {ex['output']}{tokenizer.eos_token}"}

formatted_ds = raw_ds.map(format_example, remove_columns=raw_ds.column_names)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )

tokenized_ds = formatted_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

split_ds = tokenized_ds.train_test_split(test_size=0.1, seed=42)
train_loader = DataLoader(split_ds["train"], batch_size=batch_size, shuffle=True)
val_loader = DataLoader(split_ds["test"], batch_size=batch_size, shuffle=False)

test_prompts = [
    "Задание: Объясни простыми словами, почему небо синее.\nОтвет:",
    "Задание: Напиши короткий план подготовки к экзамену по математике.\nОтвет:",
    "Задание: Придумай 3 идеи для выходных на даче зимой.\nОтвет:",
]

# baseline loss и генерации до дообучения
init_val_loss = evaluate_loss(model, val_loader, device)
print(f"\n[до обучения] Val Loss: {init_val_loss:.4f} | Val PPL: {math.exp(init_val_loss):.2f}")

before_samples = generate_samples(model, tokenizer, test_prompts, device, temperature=0.7, top_p=0.9)
for prompt, gen, _ in before_samples:
    print(f"\n--- Промпт ---\n{prompt}\n[ДО] {gen}")

#  Оптимизатор и шедулер
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
total_optim_steps = (len(train_loader) // grad_accum_steps) * epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.05 * total_optim_steps)),
    num_training_steps=total_optim_steps,
)

# Цикл обучения
print("\n=== Начало обучения ===")
global_step = 0
start_time = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss / grad_accum_steps
        loss.backward()
        running_loss += outputs.loss.item()

        if step % grad_accum_steps == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 25 == 0:
                avg_step_loss = running_loss / step
                elapsed = time.time() - start_time
                print(
                    f"Epoch {epoch}/{epochs} | Step {global_step}/{total_optim_steps} | "
                    f"Train Loss: {avg_step_loss:.4f} | Elapsed: {elapsed:.1f}s"
                )

    val_loss = evaluate_loss(model, val_loader, device)
    print(
        f">>> Конец эпохи {epoch}: Train Loss={running_loss / len(train_loader):.4f} | "
        f"Val Loss={val_loss:.4f} | Val PPL={math.exp(val_loss):.2f}"
    )

total_time = time.time() - start_time
print(f"\nОбучение завершено за {total_time:.1f} сек.")
if device.type == "cuda":
    peak_mem = torch.cuda.max_memory_allocated(device) / 1024**3
    print(f"Пиковое потребление VRAM: {peak_mem:.2f} ГБ")

# Замер генерации после дообучения на тех же промптах
after_samples = generate_samples(model, tokenizer, test_prompts, device, temperature=0.7, top_p=0.9)
for prompt, gen, _ in after_samples:
    print(f"\n--- Промпт ---\n{prompt}\n[ПОСЛЕ] {gen}")

# Сохранение чекпоинта
save_dir = "./checkpoints/rugpt3small_lab7"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\nМодель сохранена в: {save_dir}")

Using device: cuda
GPU: NVIDIA GeForce GTX 1070 | VRAM: 8.00 GB


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ru_turbo_alpaca.jsonl.zst: reconstructing file:   0%|          |  0.00B / 14.6MB            

ru_turbo_alpaca.jsonl.zst: downloading bytes:           |  0.00B            

Загружено примеров: 2000 | Колонки: ['instruction', 'input', 'output', 'label', 'all_labels', 'overlap', 'agreement', 'alternative_output']


... (2 outputs left out)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



[до обучения] Val Loss: 14.3436 | Val PPL: 1695734.98

--- Промпт ---
Задание: Объясни простыми словами, почему небо синее.
Ответ:
[ДО] Оно синее.

--- Промпт ---
Задание: Напиши короткий план подготовки к экзамену по математике.
Ответ:
[ДО] 

--- Промпт ---
Задание: Придумай 3 идеи для выходных на даче зимой.
Ответ:
[ДО] Это не проблема, это просто прикол.

=== Начало обучения ===
Epoch 1/2 | Step 25/224 | Train Loss: 8.0189 | Elapsed: 41.0s
Epoch 1/2 | Step 50/224 | Train Loss: 4.4640 | Elapsed: 79.0s
Epoch 1/2 | Step 75/224 | Train Loss: 3.2288 | Elapsed: 120.3s
Epoch 1/2 | Step 100/224 | Train Loss: 2.6137 | Elapsed: 161.9s
>>> Конец эпохи 1: Train Loss=2.4024 | Val Loss=0.7118 | Val PPL=2.04
Epoch 2/2 | Step 125/224 | Train Loss: 0.6432 | Elapsed: 202.9s
Epoch 2/2 | Step 150/224 | Train Loss: 0.6378 | Elapsed: 239.8s
Epoch 2/2 | Step 175/224 | Train Loss: 0.6375 | Elapsed: 275.4s
Epoch 2/2 | Step 200/224 | Train Loss: 0.6414 | Elapsed: 311.8s
Epoch 2/2 | Step 225/224 | Train Loss

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Модель сохранена в: ./checkpoints/rugpt3small_lab7


In [12]:
def run_temperature_experiment(model, tokenizer, device, prompt: str, temperatures=(0.2, 0.7, 1.2), max_new_tokens=70):
    print("\n" + "=" * 80)
    print("temperature")
    print(f"Промпт: {prompt!r} | max_new_tokens={max_new_tokens}")
    print("=" * 80)

    base_inputs = tokenizer(prompt, return_tensors="pt")
    prompt_len = base_inputs["input_ids"].shape[1]

    for temp in temperatures:
        set_seed(42)
        inputs = {k: v.to(device) for k, v in base_inputs.items()}

        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temp,
                top_p=0.9,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_text = tokenizer.decode(out_ids[0][prompt_len:], skip_special_tokens=True).strip()
        print(f"\n[temperature = {temp}]")
        print(gen_text)

In [20]:
def run_context_length_experiment(
    model,
    tokenizer,
    device,
    long_context_text: str,
    context_lengths=(24, 96, 220),
    temperature=0.7,
    max_new_tokens=70,
):
    print("\n" + "=" * 80)
    print("Влияние контекста")
    print(f"temperature={temperature} | max_new_tokens={max_new_tokens}")
    print("=" * 80)

    full_ids = tokenizer(long_context_text, return_tensors="pt")["input_ids"]
    total_tokens = full_ids.shape[1]
    print(f"Всего токенов в исходном тексте: {total_tokens}")

    for ctx_len in context_lengths:
        set_seed(42)
        actual_len = min(ctx_len, total_tokens)
        ctx_ids = full_ids[:, -actual_len:].to(device)
        attention_mask = torch.ones_like(ctx_ids, device=device)

        ctx_preview = tokenizer.decode(ctx_ids[0], skip_special_tokens=True)

        with torch.no_grad():
            out_ids = model.generate(
                input_ids=ctx_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.9,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        gen_text = tokenizer.decode(out_ids[0][actual_len:], skip_special_tokens=True).strip()
        print(f"\n[Длина контекста = {actual_len} токенов]")
        print(f"Хвост контекста: ...{ctx_preview[:]!r}")
        print(f"Генерация: {gen_text}")

In [21]:
ckpt_dir = "./checkpoints/rugpt3small_lab7"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(ckpt_dir, torch_dtype=torch.float32).to(device)
model.eval()
print(f"Чекпоинт загружен из {ckpt_dir} на {device}.")

prompt = "Задание: Объясни, почему небо синее.\nОтвет:"
run_temperature_experiment(
    model=model,
    tokenizer=tokenizer,
    device=device,
    prompt=prompt,
    temperatures=(0.2, 0.7, 1.2),
    max_new_tokens=70,
)

# Сравнение длины контекста:
# Начинаем с развёрнутого описания задачи и подрезаем вход до 24, 96 и 220 токенов
long_context_text = (
    "Назови 3 практических шага по исследованию малоизученного природного явления "
    "в стиле «Что? Где? Когда?». Учитывай сезонность и миграции видов, "
    "рельеф и микроклимат, влияние хищников и кормовой базы, следы "
    "жизнедеятельности на местности, а также искажение наблюдений из-за присутствия человека. "
    "Отвечай чётко, образно и по существу, без воды.\n"
    "Контекст: Наблюдатели фиксируют всплеск активности лесных зверей в сумерках, на первых порах картина кажется ясной, "
    "но затем тревожность животных растёт, а маршруты передвижения становятся непредсказуемыми"
    "Ответ:"
)
run_context_length_experiment(
    model=model,
    tokenizer=tokenizer,
    device=device,
    long_context_text=long_context_text,
    context_lengths=(24, 96, 220),
    temperature=0.7,
    max_new_tokens=70,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Чекпоинт загружен из ./checkpoints/rugpt3small_lab7 на cuda.

temperature
Промпт: 'Задание: Объясни, почему небо синее.\nОтвет:' | max_new_tokens=70

[temperature = 0.2]
Небо синее и голубое - это цвет неба в осенний день. Это значит, что оно отражает свет солнца на землю и заставляет нас чувствовать его тепло. Если вы хотите узнать больше о небе, то попробуйте рассмотреть его со всех сторон. Попробуйте увидеть его с помощью бинокля или телескопа. Оно может быть разным по цвету, форме и размеру.

[temperature = 0.7]
Солнце ярко светит в небе. Это означает, что оно находится на пути к Земле со скоростью около 30 км/с. Если вы смотрите вниз, то у вас есть возможность увидеть солнце, но если вверх, то это может вызвать проблемы с ориентацией. Во-первых, это не так красиво, как кажется сверху. Вниз нужно смотреть по горизонту

[temperature = 1.2]
Оно было голубым утром. Когда луна упала в землю на севере, оно стало зеленым днём. В небе сверкали звезды и играли с ветром белые облака. И когд

Контрольные вопросы
1. Как влияет на качество результатов длина используемого контекста? Каким параметром
трансформера она определяется?

Чем длиннее контекст, тем больше модель видит токенов, тем больше зависимостей сохраняет. При обучение больший контекст дает больше примеров зависимостей. Однако при увеличении растет и вычислительная сложность и память, а также рассеяние внимание. В самом трансформере длина контекста задаётся максимальным размером таблицы позиционных кодировок / окна внимания, через которое вычисляются позиционные эмбеддинги или маски внимания.

2. Как влияет на качество результатов число головок в механизме самовнимания?

Разные головы могут одновременно отслеживать разные типы зависимостей. Если их мало, то каждая головка вынуждена хранить много разных отношений. Если много, то некоторые головки просто избыточны: очень слабые и просто занимают ресурс.